In [ ]:
# core/engine/lib/llm/agents/_04_personal/test.ipynb

# Setup project path and log in as test user.
from neurocad.utils.testing import req
req.login_test()

In [ ]:
# Stage 1: Brief dialog (simulating user answers).

state = None
user_answers = [
    "Хочу сделать личную страницу",
    "Меня зовут Алексей Смирнов, я разработчик",
    "Мои проекты: Нейрокад и Проверка Email",
    "Люблю синие и белые цвета",
]

for msg in user_answers:
    print(f"👤 Пользователь: {msg}")
    payload = {
        "prompt": msg,
        "stage": "brief",
        "state": state,
    }
    res = req.post("/core/engine/lib/llm/agents/_04_personal/generate", payload)

    if not res.get("success"):
        print("❌ Ошибка:", res.get("error"))
        break

    data = res["data"]
    state = data.get("state")  # сервер сам ведёт историю

    if data.get("question"):
        print(f"🤖 Агент: {data['question']}")
    elif data.get("ready"):
        print("🤖 Агент: Отлично, все данные собраны! Начинаю генерацию...")
        if data.get("warning"):
            print("⚠️", data["warning"])
        break

In [ ]:
# Stage 2: Generate base layout.

from IPython.display import clear_output

if not state:
    raise RuntimeError("Сначала пройдите Stage 1 (brief) — state пуст")

payload = {
    "prompt": "Собери базовый макет по собранным данным.",
    "stage": "base",
    "state": state,
}
res = req.post("/core/engine/lib/llm/agents/_04_personal/generate", payload)

if not res.get("success"):
    print("❌ Ошибка:", res.get("error"))
else:
    data = res["data"]
    state = data["state"]           # сервер вернул обновлённый PageState
    current_html = state["html"]
    css_by_section = state["css_by_section"]
    full_css = data["css"]          # уже склеен сервером (collect_css)

    with open("preview.html", "w", encoding="utf-8") as f:
        f.write(
            "<!DOCTYPE html><html><head><meta charset='utf-8'>"
            f"<style>{full_css}</style></head><body>{current_html}</body></html>"
        )

    clear_output(wait=True)
    print("✅ Макет сгенерирован и сохранён в preview.html")
    print("   CSS по секциям: " + ", ".join(
        f"{k}={len(v)}" for k, v in css_by_section.items()
    ))

In [ ]:
# Stage 3: Add effects to ALL sections (hero, projects, footer) in one loop.

from IPython.display import clear_output

if not state or not state.get("html"):
    raise RuntimeError("Сначала пройдите Stage 2 (base) — state пуст")

# Optional per-run overrides; None → use server defaults from constants.py.
EFFECTS_MODE = "replace"   # "replace" | "append"
N_EFFECTS = 4
SEED = None

section_names = list(state["css_by_section"].keys())

for section in section_names:
    print(f"\n🎨 Генерирую эффекты для «{section}»...")

    payload = {
        "prompt": f"Добавь эффекты к секции «{section}».",
        "stage": "effects",
        "target_section": section,
        "state": state,
        "effects_mode": EFFECTS_MODE,
        "n_effects": N_EFFECTS,
        "seed": SEED,
    }
    res = req.post("/core/engine/lib/llm/agents/_04_personal/generate", payload)

    if not res.get("success"):
        print(f"❌ Ошибка для {section}: {res.get('error')}")
        continue

    data = res["data"]
    state = data["state"]          # HTML + CSS уже обновлены сервером
    print(f"✅ Эффекты для {section} добавлены.")

# Final render (display only — no business logic here).
full_css = "\n\n".join(
    v for v in state["css_by_section"].values() if v and v.strip()
)
with open("preview.html", "w", encoding="utf-8") as f:
    f.write(
        "<!DOCTYPE html><html><head><meta charset='utf-8'>"
        f"<style>{full_css}</style></head><body>{state['html']}</body></html>"
    )

clear_output(wait=True)
print("✅ Все эффекты добавлены. preview.html обновлён.")
print("   CSS по секциям: " + ", ".join(
    f"{k}={len(v)}" for k, v in state["css_by_section"].items()
))

In [ ]:
# Start a simple HTTP server to preview the generated page.

import http.server
import socketserver
import threading

PORT = 8898

# Stop a previously started server in this kernel, if any.
_prev = globals().get("_preview_httpd")
if _prev is not None:
    try:
        _prev.shutdown()
        _prev.server_close()
    except Exception:
        pass

class _Handler(http.server.SimpleHTTPRequestHandler):
    # Silence per-request logging in the notebook output.
    def log_message(self, fmt, *args):
        pass

class _ReusableTCPServer(socketserver.TCPServer):
    allow_reuse_address = True

_preview_httpd = _ReusableTCPServer(("", PORT), _Handler)
threading.Thread(target=_preview_httpd.serve_forever, daemon=True).start()

print(f"Page available at: http://127.0.0.1:{PORT}/preview.html")